In [1]:
import pandas as pd
import json
from datetime import datetime
import re

In [2]:
df = pd.read_csv("phase6.csv")
df

,extraction_id,document_type,template_id,document_path,document_name,extracted_data
0,218,SF24,1,\\192.168.1.103\nssf_phase_6_extraction_worksp...,FOLIO-5-DS07 - 10000002.jpg,"{""FORM_TYPE"": ""STANDARD CONTRIBUTIONS RETURN F..."
1,219,SF24,1,\\192.168.1.103\nssf_phase_6_extraction_worksp...,FOLIO-5-DS07 - 10000003.jpg,"{""FORM_TYPE"": ""STANDARD CONTRIBUTIONS RETURN F..."
2,220,SF24,1,\\192.168.1.103\nssf_phase_6_extraction_worksp...,FOLIO-5-DS07 - 10000004.jpg,"{""FORM_TYPE"": ""STANDARD CONTRIBUTIONS RETURN F..."
3,221,SF24,1,\\192.168.1.103\nssf_phase_6_extraction_worksp...,FOLIO-5-DS07 - 10000005.jpg,"{""FORM_TYPE"": ""STANDARD CONTRIBUTIONS RETURN F..."
4,222,SF24,1,\\192.168.1.103\nssf_phase_6_extraction_worksp...,FOLIO-5-DS07 - 10000006.jpg,"{""FORM_TYPE"": ""STANDARD CONTRIBUTIONS RETURN F..."
...,...,...,...,...,...,...
136,504,SS09,5,\\192.168.1.103\nssf_phase_6_extraction_worksp...,FOLIO-6-DS07 - 10000040.jpg,"{""MEMBER_NUMBER"": ""473275627"", ""MEMBER_NAME"": ..."
137,505,SS09,5,\\192.168.1.103\nssf_phase_6_extraction_worksp...,FOLIO-6-DS07 - 20000004.jpg,"{""MEMBER_NUMBER"": ""47304062X"", ""MEMBER_NAME"": ..."
138,506,SS09,5,\\192.168.1.103\nssf_phase_6_extraction_worksp...,FOLIO-6-DS07 - 20000005.jpg,"{""MEMBER_NUMBER"": ""49303462"", ""MEMBER_NAME"": ""..."
139,507,SS09,5,\\192.168.1.103\nssf_phase_6_extraction_worksp...,FOLIO-6-DS07 - 20000006.jpg,"{""MEMBER_NUMBER"": ""473043629"", ""MEMBER_NAME"": ..."


In [3]:

extractedJson = df["extracted_data"]
type(extractedJson)

pandas.Series

In [ ]:
parsedData = df["extracted_data"]
parsedData

In [34]:
def parseDate(date):
    dateObj = datetime.strptime(date , "%d.%m.%y")
    return dateObj.strftime("%d-%m-%Y")

def cleanNumbers(numString):
    cleanStr = "".join(re.findall(r"\d+" , numString))
    return int(cleanStr)

def clean_fund_mem_number(number):
    return re.sub(r"\." , "" , number)

def parse_amount(value):
    if value is None:
        return None

    value = str(value).strip()

    
    if value in {"", "-", "--", "/", "=", "NOT_FOUND"}:
        return None

 
    value = value.replace(",", "").replace("/-", "")
    value = re.sub(r"[=]", "", value)

    try:
        return float(value)
    except ValueError:
        return None

def process_table_rows(row):

    cleanTableRow = []

    for data in dict(row):
        print(type(data))
        table_row = {
            "AMOUNT": parse_amount(data),
            "EMPLOYEE_NAME": data["EMPLOYEE_NAME"] or None,
            "FUND_MEMBERSHIP_NUMBER": clean_fund_mem_number(data["FUND_MEMBERSHIP_NUMBER"]) or None
        }
        cleanTableRow.append(table_row)
    return cleanTableRow


sf24DataCount = 0
cleanSF24Schema = []
for _, rows in df.iterrows():
    extracted = rows["extracted_data"]
    if isinstance(extracted , str):
        extracted = json.loads(extracted)

    if extracted.get("FORM_TYPE") is None:
        extracted["FORM_TYPE"] = rows["document_type"]

    if rows["document_type"] == "SF24":
        cleanSchema = {
            "FORM_TYPE": extracted["FORM_TYPE"] or None,
            "EMPLOYER_NAME": extracted["EMPLOYER_NAME"] or None,
            "EMPLOYER_NUMBER": int(extracted["EMPLOYER_NUMBER"]) if extracted.get("EMPLOYER_NUMBER") is not None else None,
            "ADDRESS": extracted["ADDRESS"] or None,
            "BATCH_NUMBER": extracted["BATCH_NUMBER"],
            "NO_OF_ENTRIES": int(extracted["NO_OF_ENTRIES"])  if extracted.get("NO_OF_ENTRIES") is not None else None,
            "TOTAL_VALUE_SHILLINGS": extracted["TOTAL_VALUE_SHILLINGS"] or None,
            "SERIAL_NUMBER_FORM": extracted["SERIAL_NUMBER_FORM"] or None,
            "MONTH":  int(extracted["MONTH"]) if extracted.get("MONTH") is not None else None,
            "YEAR": int(extracted["YEAR"]) if extracted.get("MONTH") is not None else None,
            "CHEQUE_NUMBER": extracted["CHEQUE_NUMBER"]  if extracted.get("CHEQUE_NUMBER") is not None else None,
            "DATE_OF_SUBMISSION": parseDate("20.12.94"),
            "STANDARD_CONTRIBUTIONS_TOTAL": cleanNumbers(extracted["STANDARD_CONTRIBUTIONS_TOTAL"]) if extracted.get("CHEQUE_NUMBER") is not None else None,
            "SPECIAL_CONTRIBUTIONS_PENALTIES": extracted["SPECIAL_CONTRIBUTIONS_PENALTIES"] or None,
            "GRAND_TOTAL": extracted["GRAND_TOTAL"] or None,
            "TABLE_ROWS":  list(map(process_table_rows ,extracted["TABLE_ROWS"]))
        }
        process_table_rows(extracted["TABLE_ROWS"])
        # it = list(map(process_table_rows , extracted["TABLE_ROWS"]))
        cleanSF24Schema.append(cleanSchema)
        # print(cleanSchema)
        sf24DataCount += 1

cleanSF24Schema[13]

<class 'str'>


TypeError: string indices must be integers, not 'str'

In [15]:
clean_fund_mem_number("6.7.0.3.9.2.8.1.1")

'670392811'